In [9]:
# ============================================================
# STEP 0 — Download and prepare YOUR GitHub repository
# ============================================================

from pathlib import Path
import shutil
import urllib.request
import zipfile

REPO_URL = "https://github.com/Imvixh/flyrank-ml-internship"
ZIP_URL = "https://codeload.github.com/Imvixh/flyrank-ml-internship/zip/refs/heads/main"

ROOT = Path("/content/flyrank-ml-internship")
ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")

# Remove an incomplete/stale runtime copy
if ROOT.exists():
    shutil.rmtree(ROOT)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("Downloading YOUR repository...")
urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)

print("Extracting repository...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/content")

# GitHub extracts the repository as flyrank-ml-internship-main
EXTRACTED = Path("/content/flyrank-ml-internship-main")

# Rename to the standard path used by our notebooks
if ROOT.exists():
    shutil.rmtree(ROOT)

EXTRACTED.rename(ROOT)

# Clean up ZIP
ZIP_PATH.unlink(missing_ok=True)

# Verify repository and dataset
DATA = ROOT / "data/raw/content_refresh_anonymized.csv"

print("\n" + "=" * 55)
print("REPOSITORY CHECK")
print("=" * 55)

print("Repository exists:", ROOT.exists())
print("Dataset exists:", DATA.exists())
print("Repository:", ROOT)
print("Dataset:", DATA)

if not ROOT.exists():
    raise FileNotFoundError("Repository could not be prepared.")

if not DATA.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA}")

print("\n✓ Repository ready")
print("✓ Dataset ready")

Extracting repository...

REPOSITORY CHECK
Repository exists: True
Dataset exists: True
Repository: /content/flyrank-ml-internship
Dataset: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

✓ Repository ready
✓ Dataset ready


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Answer

**Unit of analysis:** One existing content page, identified by `content_id`.

Each row represents one content page together with its observable content, search-demand, search-performance, engagement, position, and trend signals.

**Time window:** The dataset contains page-level observations covering the available historical measurement windows in the release. The analysis uses the time-window fields provided in the dataset rather than assuming that every page has the same observation history.

The unit of analysis is therefore **one content page**, while the performance variables summarize the available historical observation windows for that page.

In [10]:
import pandas as pd

DATA = ROOT / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nUnique content_id:", df["content_id"].nunique())
print("Rows:", len(df))

print("\nOne row per content_id:",
      df["content_id"].nunique() == len(df))

print("\nDataset columns:")
print(df.columns.tolist())

Rows: 30000
Columns: 44

Unique content_id: 30000
Rows: 30000

One row per content_id: True

Dataset columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Answer

**Features:**  
Search volume, competition, competition level, CPC, content type, main intent, word count, character count, content age, days since last update, impressions, clicks, sessions, CTR, average position, engagement rate, scroll rate, AI traffic percentage, and the available tier/bucket fields derived from these measurements.

**Label:**  
`trend_direction`, used to define the observed decline outcome for the modeling task.

**Context:**  
`content_id` and `client_id` are identifiers used to trace records and group observations. They are context rather than predictive features.

**Excluded:**  
Fields that directly encode the outcome or a post-outcome decision should not be used as predictive features. `trend_direction` and `trend_pct` are excluded from the feature vector when predicting decline because they define or directly describe the outcome. Identifiers are also excluded from model features to avoid memorization and leakage.

In [11]:
# Section 2 — Verify the fields used by the data contract

feature_candidates = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

context_fields = ["content_id", "client_id"]

excluded_fields = ["trend_direction", "trend_pct"]

print("FEATURE FIELDS")
print(feature_candidates)

print("\nCONTEXT FIELDS")
print(context_fields)

print("\nEXCLUDED FIELDS")
print(excluded_fields)

print("\nFields missing from dataset:")
all_planned = feature_candidates + context_fields + excluded_fields
print([c for c in all_planned if c not in df.columns])

FEATURE FIELDS
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

CONTEXT FIELDS
['content_id', 'client_id']

EXCLUDED FIELDS
['trend_direction', 'trend_pct']

Fields missing from dataset:
[]


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Answer

The data contract is verified by checking the dataset grain, record counts, missing values, and the available time-window fields. The expected grain is one row per `content_id`. The checks below verify that assumption and identify important missingness and historical-window information before modeling.

In [12]:
# Section 3 — Verify grain, counts, missing values, and windows

print("=== GRAIN CHECK ===")
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Duplicate content_id rows:", df["content_id"].duplicated().sum())

print("\n=== MISSING VALUES ===")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_count", ascending=False)

display(missing_table[missing_table["missing_count"] > 0])

print("\n=== TIME / WINDOW RELATED FIELDS ===")
window_fields = [
    c for c in df.columns
    if any(term in c.lower() for term in ["90d", "days", "age", "update"])
]

print(window_fields)

print("\n=== TREND LABEL COUNTS ===")
print(df["trend_direction"].value_counts(dropna=False))

print("\n=== CONTRACT CHECK ===")
assert df["content_id"].nunique() == len(df), "Duplicate content_id values found."

print("✓ One row per content_id verified")
print("✓ Record count verified")
print("✓ Missing values inspected")
print("✓ Window-related fields identified")

=== GRAIN CHECK ===
Rows: 30000
Unique content_id: 30000
Duplicate content_id rows: 0

=== MISSING VALUES ===


,missing_count,missing_pct
provider_used,21438,71.46
word_count,7699,25.66
char_count,7699,25.66
word_count_tier,7699,25.66
char_count_tier,7699,25.66
model_used,5733,19.11
trend_pct,3388,11.29
competition_level,2610,8.70
search_volume,2468,8.23
cpc,2468,8.23



=== TIME / WINDOW RELATED FIELDS ===
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'engagement_rate']

=== TREND LABEL COUNTS ===
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

=== CONTRACT CHECK ===
✓ One row per content_id verified
✓ Record count verified
✓ Missing values inspected
✓ Window-related fields identified


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Answer

This dataset has several important limits.

First, the historical observation windows are summarized at page level, so the data does not provide a complete balanced time series for every page. Some fields contain substantial missingness, including content-size fields and provider/model fields.

Second, the available 90-day metrics describe observed historical performance, but they do not establish a causal relationship between a content change and a later performance change.

Third, the dataset contains overlapping historical measurements such as impressions, clicks, sessions, and engagement. These are useful observed signals, but their timing must be considered carefully when building a predictive model to avoid using information that overlaps with the outcome definition.

Finally, the dataset does not contain all factors that influence search performance, such as every search-engine ranking signal, competitor actions, algorithm changes, or complete business outcomes. Therefore, the analysis should be interpreted as directional decision support rather than a complete explanation of why a page's performance changes.

In [13]:
# Section 4 — Verify important data limitations

print("=== MISSINGNESS LIMITS ===")

key_missing = [
    "word_count",
    "char_count",
    "search_volume",
    "cpc",
    "competition",
    "main_intent",
    "trend_pct",
    "scroll_rate",
]

for col in key_missing:
    if col in df.columns:
        pct = df[col].isna().mean() * 100
        print(f"{col}: {pct:.2f}% missing")

print("\n=== HISTORICAL WINDOW FIELDS ===")

window_fields = [
    c for c in df.columns
    if "90d" in c.lower()
]

print("90-day fields:", len(window_fields))
print(window_fields)

print("\n=== TREND DISTRIBUTION ===")
trend_counts = df["trend_direction"].value_counts(dropna=False)
display(trend_counts.to_frame("count"))

print("\n✓ Data limitations checked")
print("✓ Missingness checked")
print("✓ Historical-window fields checked")
print("✓ Outcome distribution checked")

=== MISSINGNESS LIMITS ===
word_count: 25.66% missing
char_count: 25.66% missing
search_volume: 8.23% missing
cpc: 8.23% missing
competition: 8.23% missing
main_intent: 7.91% missing
trend_pct: 11.29% missing
scroll_rate: 0.42% missing

=== HISTORICAL WINDOW FIELDS ===
90-day fields: 8
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']

=== TREND DISTRIBUTION ===


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



✓ Data limitations checked
✓ Missingness checked
✓ Historical-window fields checked
✓ Outcome distribution checked


## Self-check — VERIFIED

- [x] Every section above is filled — markdown thinking and supporting code are complete.
- [x] The notebook runs top to bottom with no errors — verified with Runtime → Run all.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful words such as observed, measured, directional, and decision-support.
- [x] Repository and dataset checks passed successfully.
- [x] The notebook is ready to be committed to my repository under `work/notebooks/`.